# CYTools-agent

An agent that drives CYTools (fetch polytopes, triangulate, build CYs) with a local Ollama model.

Run with the **Python (cytools-agent)** kernel. Run the **Setup** cells once, then **chat** -- add `agent.chat(...)` cells freely; the agent remembers earlier turns. Re-run the *Start a session* cell to reset.

## Setup (run once)

In [ ]:
from openai import OpenAI
import os

base = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
client = OpenAI(base_url=base + "/v1", api_key="ollama")
MODEL = "qwen3:14b"   # a bigger local model behaves noticeably cleaner than 8b;
                      # drop to "qwen3:8b" if the point is "even a tiny model works"
assert MODEL in [m.id for m in client.models.list().data], f"{MODEL} not pulled"
print("OK:", MODEL)

In [2]:
from cytools_agent.tools import (polytope, triangulation, cy, code,
                                 mapping, glossary)
from cytools_agent.schema import function_to_schema

TOOL_FNS = [polytope.fetch_polytopes, polytope.get_polytope_info,
            polytope.ks_stats,
            triangulation.get_heights,
            triangulation.get_triangulation_info,
            cy.get_cy_info, cy.get_cy_cones,
            mapping.compute_for_each, mapping.make_plot,
            mapping.search_polytopes,
            glossary.cy_glossary, glossary.reference,
            code.run_python, code.cytools_help]
tools = [function_to_schema(fn) for fn in TOOL_FNS]
tool_impls = {fn.__name__: fn for fn in TOOL_FNS}

In [3]:
from cytools_agent.agent import Agent
from cytools_agent.prompt import DEFAULT_SYSTEM_PROMPT as system_prompt
# (edit system_prompt here to customize)

## Start a session

Run this to begin -- or re-run it to reset the conversation.

In [4]:
agent = Agent(client, MODEL, system_prompt, tools, tool_impls,
              max_steps=20, verbosity=2)

## Chat

Add more `agent.chat(...)` cells below; the agent remembers earlier turns.

In [5]:
print(agent.chat("Fetch 3 polytopes at h11=5"))

  -> ks_stats(h11=5)
  -> fetch_polytopes(limit=3, h11=5)

The three polytopes with h11=5 are:  
`h11-5_h21-20_ind-0`, `h11-5_h21-29_ind-0`, and `h11-5_h21-29_ind-1`.


In [6]:
print(agent.chat("How many NTFEs do each of them have?"))

  -> compute_for_each(ks_inds=['h11-5_h21-20_ind-0', 'h11-5_h21-29_ind-0', 'h11-5_h21-2..., expressions={'ntfe_count': "get_heights(ks_ind, kind='NTFE')['shape']...)

The three polytopes have the following NTFE counts:  
- `h11-5_h21-20_ind-0`: **2** NTFEs  
- `h11-5_h21-29_ind-0`: **1** NTFE  
- `h11-5_h21-29_ind-1`: **1** NTFE


In [7]:
print(agent.chat("What are the CY volumes at the tip of the stretched kahler cone for each of their associated CYs?"))

  -> compute_for_each(expressions={'cy_volume': "get_cy_info(ks_ind, get_heights(ks_ind, ki..., ks_inds=['h11-5_h21-20_ind-0', 'h11-5_h21-29_ind-0', 'h11-5_h21-2...)

The CY volume at the stretched Kahler cone tip is computable only for favorable polytopes. Here's the result:

- **`h11-5_h21-20_ind-0`**: CY volume = **33.604167** (valid, favorable)  
- **`h11-5_h21-29_ind-0`**: ❌ Non-favorable (error: "unsupported CY")  
- **`h11-5_h21-29_ind-1`**: ❌ Non-favorable (error: "unsupported CY")  

To get volumes for the latter two, fetch **favorable** polytopes via `fetch_polytopes(..., favorable=True)`.


## Bigger questions (iterate and plot)

The same `agent.chat` handles multi-polytope questions and figures. It uses the harness-side tools -- `compute_for_each` (one expression per id, stored as aligned lists), `make_plot`, and `search_polytopes` -- instead of looping by hand. Saved figures land under `notebooks/scratch/`.

In [ ]:
print(agent.chat(
    "For the first 10 polytopes at h11=3, compute each one's NTFE triangulation "
    "count and scatter it against h21."))

In [ ]:
# Save the session as a standalone Python script:
# agent.save_history("session.py")